In [9]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

In [10]:
from app.config import settings

In [11]:
model_name = settings.EMBEDDING_MODEL
dimension = settings.EMBEDDING_DIMENSION
client = GoogleGenerativeAIEmbeddings(
            model=model_name,
            google_api_key=settings.GEMINI_API_KEY,
        )

In [12]:
client.embed_query("hi how are you?")

[-0.023745231330394745,
 0.0083352355286479,
 0.004480163101106882,
 -0.08734215050935745,
 -0.005805687978863716,
 -0.004170320928096771,
 -0.010113386437296867,
 0.0020749305840581656,
 0.009593961760401726,
 0.007590307854115963,
 -0.00020999573462177068,
 -0.011602381244301796,
 0.003844328923150897,
 0.008087096735835075,
 0.1572096049785614,
 -0.007285694591701031,
 0.007966679520905018,
 0.0067625935189425945,
 0.011660374701023102,
 -0.02323327399790287,
 0.002747324062511325,
 -0.000856351456604898,
 -0.0038450937718153,
 0.0055117253214120865,
 -0.004431517329066992,
 0.01339320745319128,
 0.011473800987005234,
 -0.0029700114391744137,
 0.014682948589324951,
 -0.004070257768034935,
 0.008310345932841301,
 -0.003404349321499467,
 -0.0012621929636225104,
 0.013595560565590858,
 0.0074196248315274715,
 0.005093727260828018,
 0.009267681278288364,
 0.01972142420709133,
 -0.00875735655426979,
 0.003038765164092183,
 0.012902314774692059,
 0.00021392149210441858,
 -0.01437191013246

In [2]:
from app.ingestion.processor import IngestionProcessor

d:\Projects\RAG_Systems\DriveRAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
ingest = IngestionProcessor()

In [1]:
import socket
import os

host = "generativelanguage.googleapis.com"

try:
    print("DNS:", socket.getaddrinfo(host, 443)[0][4])
except OSError as e:
    print("DNS failed:", repr(e))

print("Proxy variables:", {
    name: os.environ.get(name)
    for name in ("HTTPS_PROXY", "HTTP_PROXY", "ALL_PROXY", "NO_PROXY")
})


DNS: ('172.217.112.4', 443)
Proxy variables: {'HTTPS_PROXY': None, 'HTTP_PROXY': None, 'ALL_PROXY': None, 'NO_PROXY': None}


In [28]:
vector = client.embed_query("سلام")
print("Embedding OK — dimensions:", len(vector))

Embedding OK — dimensions: 3072


In [26]:
from pathlib import Path

ingest.ingest_file(
    Path("DATA/true_data/parallel_work_queue.txt"),
    source_type="txt",
)

ResponseHandlingException: [WinError 10061] No connection could be made because the target machine actively refused it

In [27]:
import traceback

try:
    ingest.ingest_file(
        Path("DATA/true_data/parallel_work_queue.txt"),
        source_type="txt",
    )
except Exception as e:
    print("Exception:", type(e).__name__, str(e))
    print("\nآخرین فراخوانی‌های traceback:")
    for frame in traceback.extract_tb(e.__traceback__)[-12:]:
        print(f"{frame.filename}:{frame.lineno} — {frame.name}")


Exception: ResponseHandlingException [WinError 10061] No connection could be made because the target machine actively refused it

آخرین فراخوانی‌های traceback:
d:\Projects\RAG_Systems\DriveRAG\app\ingestion\processor.py:115 — _ensure_collection
d:\Projects\RAG_Systems\DriveRAG\.venv\Lib\site-packages\qdrant_client\qdrant_client.py:1609 — collection_exists
d:\Projects\RAG_Systems\DriveRAG\.venv\Lib\site-packages\qdrant_client\qdrant_remote.py:1935 — collection_exists
d:\Projects\RAG_Systems\DriveRAG\.venv\Lib\site-packages\qdrant_client\http\api\collections_api.py:438 — collection_exists
d:\Projects\RAG_Systems\DriveRAG\.venv\Lib\site-packages\qdrant_client\http\api\collections_api.py:67 — _build_for_collection_exists
d:\Projects\RAG_Systems\DriveRAG\.venv\Lib\site-packages\qdrant_client\http\api_client.py:98 — request
d:\Projects\RAG_Systems\DriveRAG\.venv\Lib\site-packages\qdrant_client\http\api_client.py:115 — send
d:\Projects\RAG_Systems\DriveRAG\.venv\Lib\site-packages\qdrant_clien

In [29]:
try:
    print("Qdrant reachable:", ingest.qdrant_client.get_collections())
except Exception as e:
    print(type(e).__name__, str(e))

ResponseHandlingException [WinError 10061] No connection could be made because the target machine actively refused it


In [30]:
print(settings.QDRANT_CLUSTER_ENDPOINT)

https://e2c3824c-e709-4f36-8e77-7e6382343038.europe-west3-0.gcp.cloud.qdrant.io


In [4]:
from urllib.parse import urlparse
from app.config import settings

endpoint = str(settings.QDRANT_CLUSTER_ENDPOINT).strip()
parsed = urlparse(endpoint)

print("Scheme:", parsed.scheme)
print("Host:", parsed.hostname)
print("Port:", parsed.port)
print("Collection:", settings.QDRANT_COLLECTION)


Scheme: https
Host: e2c3824c-e709-4f36-8e77-7e6382343038.europe-west3-0.gcp.cloud.qdrant.io
Port: None
Collection: DriveRAG


In [5]:
import socket
from urllib.parse import urlparse
from app.config import settings

host = urlparse(str(settings.QDRANT_CLUSTER_ENDPOINT).strip()).hostname

print("Host present:", bool(host))
if host:
    try:
        with socket.create_connection((host, 443), timeout=10):
            print("TCP 443: connected")
    except OSError as e:
        print("TCP 443:", type(e).__name__, str(e))


Host present: True
TCP 443: connected


In [6]:
from qdrant_client import QdrantClient
from app.config import settings

client = QdrantClient(
    url=str(settings.QDRANT_CLUSTER_ENDPOINT).strip(),
    api_key=settings.QDRANT_API_KEY,  # اگر نام تنظیمات کلید در پروژه‌ات متفاوت است، اصلاحش کن
    prefer_grpc=False,
    timeout=15,
)

try:
    print(client.get_collections())
except Exception as e:
    print(type(e).__name__, str(e))


collections=[CollectionDescription(name='DriveRAG')]


In [4]:
from pathlib import Path
from app.ingestion.processor import IngestionProcessor

ingest = IngestionProcessor()
file_path = Path("DATA/true_data/pods_autoscale.html")

ingest.ingest_file(file_path, source_type="true_data")


10:19:48.865 Processing File
10:19:48.866   HTML Parsing
10:19:48.902   ✂️ Text Chunking (LangChain)
10:19:48.912     ✅ Generated 28 chunks using LangChain
10:19:48.918   Saved processed data to D:\Projects\RAG_Systems\DriveRAG\data\processed\true_data\pods_autoscale.html.json
10:19:48.919   Vectorizing & Indexing
10:20:21.889     Indexed pods_autoscale.html: chunks 0..4
10:20:43.125     Indexed pods_autoscale.html: chunks 5..9
10:21:05.431     Indexed pods_autoscale.html: chunks 10..14
10:21:28.222     Indexed pods_autoscale.html: chunks 15..19
10:21:50.215     Indexed pods_autoscale.html: chunks 20..24
10:21:55.730     Indexed pods_autoscale.html: chunks 25..27
10:21:55.731   Successfully indexed pods_autoscale.html


In [2]:
import logfire

logfire.configure()
logfire.info("DriveRAG Logfire test")
logfire.force_flush()

Logfire project URL: https://logfire-us.pydantic.dev/mhmdrza9789/drive-rag

18:56:44.317 DriveRAG Logfire test


True

In [5]:
from pathlib import Path
from app.ingestion.processor import IngestionProcessor

root = Path("DATA/true_data")
print("Folder exists:", root.is_dir())

ingest = IngestionProcessor()
ingest.ingest_directory(root, source_type="true_data")


Folder exists: True
10:27:31.959 Ingest Directory
10:27:31.960   Processing File
10:27:32.330     ✂️ Text Chunking (LangChain)
10:27:32.331       ✅ Generated 27 chunks using LangChain
10:27:32.333     Saved processed data to D:\Projects\RAG_Systems\DriveRAG\data\processed\true_data\architecture.pptx.json
10:27:32.334     Vectorizing & Indexing
10:27:40.383       Indexed architecture.pptx: chunks 0..4
10:27:46.811       Indexed architecture.pptx: chunks 5..9
10:27:53.214       Indexed architecture.pptx: chunks 10..14
10:27:59.580       Indexed architecture.pptx: chunks 15..19
10:28:05.957       Indexed architecture.pptx: chunks 20..24
10:28:07.003       Indexed architecture.pptx: chunks 25..26
10:28:07.004     Successfully indexed architecture.pptx
10:28:07.005   Processing File
10:28:07.369     ✂️ Text Chunking (LangChain)
10:28:07.372       ✅ Generated 54 chunks using LangChain
10:28:07.376     Saved processed data to D:\Projects\RAG_Systems\DriveRAG\data\processed\true_data\cronjobs.

In [8]:
from app.services.retrieval.qdrant_service import search_enterprise_knowledge

query = "what is Kubernetes?"

results = search_enterprise_knowledge(
    query=query,
    limit=8,
)

print("Result count:", len(results))

for index, result in enumerate(results, start=1):
    print(f"\n--- Result {index} ---")
    print("Score:", result.get("score"))
    print("Source:", result.get("source"))
    print("Content length:", len(result.get("content", "")))
    print("Content preview:")
    print(result.get("content", "")[:500])


19:13:12.398 Qdrant search completed
Result count: 8

--- Result 1 ---
Score: 0.87158585
Source: architecture.pptx
Content length: 313
Content preview:
--- Slide 5 ---
Intro - What Does Kubernetes do?
Kubernetes is the linux kernel of distributed systems.
It abstracts away the underlying hardware of the nodes and provides a  uniform interface for applications to be both deployed and consume the  shared pool of resources.

--- Slide 6 ---
Kubernetes
Architecture

--- Result 2 ---
Score: 0.86615705
Source: architecture.pptx
Content length: 770
Content preview:
--- Slide 1 ---
Kubernetes
A Comprehensive Overview
Kubernetes v1.8

--- Slide 2 ---
Agenda
Introduction
Who am I?
What is Kubernetes?
What does Kubernetes do?
Architecture
Master Components
Node Components
Additional Services
Networking
Concepts
Core
Workloads
Network
Storage
Configuration
Auth and Identity
Behind the Scenes
Deployment from Beginning to End

--- Slide 3 ---
Introduction

--- Slide 4 ---
Intro - What is Kubernetes?

In [4]:
from app.config import settings

print("Embedding model:", settings.EMBEDDING_MODEL)
print("Embedding dimension:", settings.EMBEDDING_DIMENSION)
print("Gemini key configured:", bool(settings.GEMINI_API_KEY))
print("Gemini key length:", len(settings.GEMINI_API_KEY or ""))
print("Qdrant endpoint configured:", bool(settings.QDRANT_CLUSTER_ENDPOINT))

Embedding model: models/gemini-embedding-001
Embedding dimension: 3072
Gemini key configured: True
Gemini key length: 53
Qdrant endpoint configured: True


In [7]:
from app.services.retrieval.embedding import GeminiEmbeddingProvider

provider = GeminiEmbeddingProvider()

try:
    vector = provider.embed_query(" what is Kubernetes ?")
    print("Embedding succeeded")
    print("Vector type:", type(vector))
    print("Vector length:", len(vector))
    print("First values:", vector[:5])
except Exception:
    import traceback
    traceback.print_exc()


Embedding succeeded
Vector type: <class 'list'>
Vector length: 3072
First values: [-0.02085071988403797, -0.00050501519581303, -0.00024043102166615427, -0.07063840329647064, -0.0024853376671671867]
